# OpenVLA transfer arm — Colab T4 probe

Runs the **robot-pretrained OpenVLA-7B transfer arm** in 4-bit on a **free Colab T4**.
The 50-step smoke (2026-07-16) already proved it loads and trains without OOM/dtype errors.
This is now a **600-step probe (~50 min)** answering the question the smoke could not:
**does the pipeline learn anything from the camera at all**, before you book the lab GPU?

**Scope / what this is NOT.** Still not a result. The `scratch` and `prismatic` control arms
need >=24 GB bf16 and will **not** fit a T4; run the real 3-arm ablation on the lab GPU with
`configs/openvla.yaml`.

**How to read the loss.** The trainer prints a **marginal-only loss floor** (2.563 nats for
waypoint) — what a model scores knowing the action distribution but ignoring the image.
Breaking below it means the model is using the camera; settling at it means it is not.
The 50-step runs both sat ~0.1 nats off their floor, which is all 200 samples buys.

`velocity` mode is ill-posed from a single frame (motion is invisible in one image), so every
arm converges to the marginal and the ablation returns a null result by construction. That is
why this config uses `waypoint`. **Do not switch it back.**

### Before you start
1. **Runtime -> Change runtime type -> T4 GPU.**
2. **Push the repo first, and re-run cell 2 on every retry.** This notebook clones `main`,
   so any fix must be pushed *before* the clone. Re-running only the train cell in a live
   session silently reuses the OLD clone — that is exactly how the 15:18 run ended up with
   new data but stale logging code.
3. **Upload the data to Google Drive.** `data/uzh_fpv/` is git-ignored, so it travels via
   Drive. It must be data converted at or after commit `6789b8f` — earlier conversions
   clipped velocity at 5 m/s (saturating a 13 m/s dataset) and carry no `poses.npy`, which
   waypoint mode requires. Rebuild it on your laptop with:
   ```bash
   python scripts/convert_uzh_fpv.py --src data/raw/uzh_fpv --dst data/uzh_fpv
   tar czf uzh_fpv.tar.gz -C data uzh_fpv
   ```
   then upload `uzh_fpv.tar.gz` to Drive at `MyDrive/alpamayo/uzh_fpv.tar.gz`, **deleting
   the old copy first** (a stale one is indistinguishable by name and silently reproduces
   the saturated run). Adjust the path in the data cell if you put it elsewhere.

## 1. Confirm the GPU is a T4
Turing (T4) / Pascal (P100) have no bf16 tensor cores — that's why this run uses fp16.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo

In [ ]:
%cd /content
!rm -rf alpamayo_drone
!git clone https://github.com/AnuragMandke/alpamayo_drone.git
%cd /content/alpamayo_drone
!git log --oneline -1

## 3. Install the OpenVLA env deps
OpenVLA's `trust_remote_code` needs **transformers 4.40.1** (Colab ships a much newer one, so
this downgrades it). `accelerate` and `peft` are **pinned to that same era, not floored** —
Colab preinstalls versions that satisfy a `>=` floor, and a modern accelerate moves 4-bit
models with `.to()`, which transformers 4.40.1 forbids (it crashes *after* the 15 GB load).

We deliberately **do not** reinstall torch/torchvision — Colab's preinstalled build is matched
to the runtime CUDA, and replacing it from PyPI often breaks the GPU.

The version print below is the check that the downgrade actually took effect. **If it shows
anything else, restart the session** — pip cannot replace an already-imported module.

In [ ]:
!pip install -q \
  "transformers==4.40.1" "tokenizers>=0.19,<0.20" "timm==0.9.10" \
  "accelerate==0.29.3" "peft==0.11.1" "bitsandbytes>=0.43.0" \
  "Pillow>=10.0.0" "PyYAML>=6.0"

import transformers, accelerate, peft, bitsandbytes
print(f'transformers={transformers.__version__} accelerate={accelerate.__version__} '
      f'peft={peft.__version__} bitsandbytes={bitsandbytes.__version__}')
print('Expect transformers=4.40.1 accelerate=0.29.3 peft=0.11.1. If not, the old versions '
      'are still imported: Runtime -> Restart session, then re-run from cell 2.')

## 4. Bring the data over from Drive
Mounts Drive and extracts the tarball you uploaded into `data/uzh_fpv/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

TARBALL = '/content/drive/MyDrive/alpamayo/uzh_fpv.tar.gz'  # <-- edit if you used another path
!mkdir -p data
!tar xzf "$TARBALL" -C data
!echo 'trajectories:' && ls data/uzh_fpv/trajectories | wc -l && ls data/uzh_fpv/trajectories | head -3

## 5. Point the HF cache at local disk
OpenVLA weights (~15 GB) download here. This is on Colab's fast ephemeral disk and is
**re-downloaded every session** — a free 15 GB Drive can't hold it, so we don't persist it.

In [ ]:
import os
os.environ['HF_HOME'] = '/content/hf_cache'

## 6. Run the probe (~50 min)
First run spends several minutes downloading the ~15 GB of weights, then trains **600 steps**
(~50 min at ~12 steps/min on a T4). The `pretrained` arm is the only one that fits a 16 GB T4
(4-bit QLoRA).

Watch the `[Train] marginal-only loss floor = ...` line printed at startup — the run is read
against it. Keep the tab active; a free-tier disconnect costs the whole 15 GB download.

In [ ]:
!python scripts/train_openvla.py --config configs/openvla_colab.yaml --init pretrained

## What success looks like
- `[Train] precision=torch.float16 (autocast=on)` and `[Train] target_mode=waypoint horizon=8`
  near the top. If it says `target_mode=velocity`, your clone is stale — re-run cell 2.
- `[Train] marginal-only loss floor = 2.563 nats` — the trainer computes this from your
  actual train split. **This is the number the whole run is read against.**
- The `loss=` line ends with `(mean of last N batches)`. If it does not, you are running
  pre-`6789b8f` code and the number is a *cumulative* mean that slides downward on its own.
- Trainable-params line shows only the LoRA adapters (33.5M / 7.57B = 0.44%).
- Ends with `[max_steps=600 reached — stopping]` and saves
  `outputs/openvla/pretrained/epoch001` (~130 MB — adapters only).

### The number that matters
**The printed marginal floor (2.563) is what a model scores knowing the action distribution
but ignoring the image entirely.** This 600-step run (~50 min, 2400 samples) exists to answer
one question: does the loss break below it?

- **Breaks below (say ~2.3 or lower)** — the model is using the camera. The pipeline works;
  book the lab GPU for the 3-arm ablation.
- **Stalls at ~2.56** — it has learned the action prior and nothing about what it sees, and
  that is worth knowing on a free T4 rather than after booking the GPU. Prime suspect is the
  attention-only LoRA (`LORA_TARGETS` = q/k/v/o_proj), where OpenVLA's official recipe uses
  `all-linear` including the MLP.

For reference, both 50-step runs landed ~0.1 nats below their floor (velocity 2.450 vs 2.576;
waypoint 2.482 vs 2.563) — i.e. essentially at the marginal. That was expected: 200 samples
buys the marginal and little else, which is exactly why this longer run exists.

Note the reported loss is diluted ~2x: it averages 8 supervised positions, but
`drone_to_openvla` writes constant zeros into roll/pitch/gripper, so 3 of the 7 action
tokens plus EOS are free to predict. Only dims `[0,1,2,5]` carry drone data — the offline
eval should score those 4 alone.

## Troubleshooting
- **CUDA OOM** — lower `batch_size` to 2 (or 1) in `configs/openvla_colab.yaml`.
- **`loss=nan`** — fp16 underflow; the GradScaler should handle it, but if it persists
  drop `optimizer.lr` and/or `batch_size`.
- **`FileNotFoundError: poses.npy`** — waypoint mode needs poses; your Drive tarball predates
  commit `6789b8f`. Re-convert and re-upload (see the header).
- **`` `.to` is not supported for `4-bit` ``, at the end of `from_pretrained`** — your
  `accelerate` is newer than `transformers==4.40.1`. Re-run the install cell (it pins
  `accelerate==0.29.3`) and **restart the session**; check the version print.
- **bf16 error inside the OpenVLA remote code** — the remote `modeling_prismatic.py` may
  hardcode a bf16 path in the vision tower. If so, that's exactly the finding this smoke
  test exists to surface — report it; the fix is a small patch to force the vision dtype.
- **Session disconnects mid-run** — free Colab idles out and this run is ~50 min; keep the
  tab active. The weights (~15 GB) re-download each session, so a disconnect is expensive.